# 🏢 노바코프 내규 챗봇 만들기 — Day 1 실습

---

## 🎯 오늘의 미션

> **노바코프(Novacorp)** 에 인턴으로 입사했습니다.  
> 팀장님이 첫 번째 미션을 주셨습니다.
>
> *"직원들이 내규 관련 질문을 인사팀에 너무 많이 해요.*  
> *오늘 배운 내용만 써서 챗봇 프로토타입을 만들어주세요."*
>
> 오늘 배운 것: **EXAONE 로컬 추론 · 파라미터 · 멀티턴**  
> 목표: **노바코프 내규를 아는 챗봇 v0.1 완성**

---

## 🗺️ 구현 단계

```
STAGE 1. 모델 로드   — EXAONE을 내 GPU에 불러오기
STAGE 2. 역할 부여   — 노바코프 전문가로 만들기
STAGE 3. 대화 기억   — 이전 대화를 기억하게 하기
STAGE 4. 안전 장치   — 실패해도 괜찮게 만들기
STAGE 5. 챗봇 완성   — 실제로 대화해보기
```

> 💡 각 단계마다 **결과를 직접 확인**하며 나아갑니다.  
> 단계가 끝날 때마다 ✅ 표시가 나와야 다음으로 넘어가세요!

---
## ⚙️ 준비 — 패키지 설치 & 임포트

먼저 필요한 도구들을 불러옵니다.  
이 셀은 **수정 없이 실행**만 하세요.

In [ ]:
# 필요한 라이브러리 설치 (처음 1회만)
!pip install -q --upgrade transformers accelerate

In [ ]:
import os
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("✅ 준비 완료!")

---
# STAGE 1. 모델 로드 — EXAONE을 내 GPU에 불러오기

챗봇을 만들기 전에, 먼저 **EXAONE 모델이 정상적으로 로드되는지** 확인합니다.  
건물 공사 전에 전기가 들어오는지 확인하는 것과 같습니다.

### 1-1. GPU 확인

EXAONE은 GPU가 있어야 실행됩니다. 먼저 GPU 상태를 점검합니다.

> ⚠️ **GPU가 연결되어 있지 않으면 아래 단계가 작동하지 않습니다!**  
> Colab 메뉴 → **런타임 → 런타임 유형 변경 → T4 GPU** 로 설정하세요.

In [ ]:
# GPU 상태 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

### 1-2. 토크나이저 & 모델 로드

HuggingFace Hub에서 EXAONE 모델을 다운로드하고 GPU에 올립니다.

> ⚠️ 처음 다운로드는 약 **2~3분** 소요됩니다. ReadTimeout이 발생하면 해당 셀만 다시 실행하세요.

In [ ]:
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
model_name = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"

print("📥 토크나이저 로드 중...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
print("✅ STAGE 1-1 완료: 토크나이저 로드 성공")

In [ ]:
print("📥 모델 로드 중... (약 2~3분 소요)")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto",
)
print(f"✅ STAGE 1-2 완료: 모델 로드 성공!")
print(f"   GPU 메모리: {torch.cuda.memory_allocated() / 1024**3:.1f} GB 사용 중")

### 1-3. 추론 함수 & 토큰 추적기 준비

`generate_response` 함수와 토큰 추적기를 만들어둡니다.  
이후 모든 호출에서 이 함수와 추적기를 사용합니다.

In [ ]:
# 추론 함수 — 메시지 리스트를 받아 응답 dict를 반환
def generate_response(messages, max_new_tokens=300, temperature=0.3, top_p=0.9):
    """EXAONE 로컬 추론 — 메시지 리스트를 받아 응답 dict를 반환한다."""
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    input_ids = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    ).input_ids.to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True if temperature > 0 else False,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][input_ids.shape[1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return {
        "content": answer,
        "input_tokens": input_ids.shape[1],
        "output_tokens": len(generated_ids),
    }


# 토큰 추적기 — 모든 호출의 토큰을 자동으로 합산
class TokenTracker:
    def __init__(self):
        self.total_input  = 0   # 입력 토큰 누계
        self.total_output = 0   # 출력 토큰 누계
        self.call_count   = 0   # 호출 횟수

    def add(self, result):     # generate_response 결과를 받아 자동 집계
        self.total_input  += result["input_tokens"]
        self.total_output += result["output_tokens"]
        self.call_count   += 1

    def report(self):          # 지금까지 사용한 토큰 출력
        api_cost = (self.total_input * 0.15 + self.total_output * 0.60) / 1_000_000
        print(f"📊 총 {self.call_count}회 호출")
        print(f"   입력 {self.total_input:,}토큰 + 출력 {self.total_output:,}토큰")
        print(f"   로컬 추론 비용: ₩0")
        print(f"   (참고) API 환산 비용: ${api_cost:.4f} (약 {api_cost * 1400:.1f}원)")


tracker = TokenTracker()
print("✅ STAGE 1-3 완료: 추론 함수 & 토큰 추적기 준비!")

### 1-4. 첫 번째 추론 테스트

모델이 제대로 작동하는지 **간단한 질문 하나**로 확인합니다.

In [ ]:
# 연결 확인용 테스트 호출
test_result = generate_response(
    messages=[{"role": "user", "content": "안녕하세요! 연결 테스트입니다. 한 문장으로 인사해주세요."}],
    temperature=0.3,
    max_new_tokens=50,
)

# 응답 출력
print("EXAONE 응답:", test_result["content"])
print(f"입력 토큰: {test_result['input_tokens']} / 출력 토큰: {test_result['output_tokens']}")

tracker.add(test_result)
print("\n✅ STAGE 1-4 완료: EXAONE 추론 확인!")

---
# STAGE 2. 역할 부여 — 노바코프 전문가로 만들기

EXAONE은 기본적으로 "무엇이든 아는 AI" 입니다.  
**system 프롬프트**로 "노바코프 내규 전문가" 로 역할을 좁혀줍니다.

### 2-1. 노바코프 내규 데이터 준비

챗봇이 답변할 때 참고할 **핵심 내규**를 정리합니다.

In [ ]:
# 노바코프 핵심 내규 — 챗봇이 이 내용을 바탕으로 답변합니다
NOVACORP_RULES = """
[연차/휴가]
- 정규직: 입사 1년 후 연 15일 유급 연차 / 5년 이상 20일 / 10년 이상 25일
- 연차 신청: 사용일 7일 전까지 부서장 승인 필요
- 병가: 연 10일 유급 (3일 이상은 진단서 필요)
- 결혼 휴가: 본인 5일 / 출산휴가: 90일 (쌍둥이 120일)

[재택근무]
- 주 2일까지 가능 / 3일 전 부서장 승인 필요
- 오전 9시 30분까지 업무 시작 보고 / 회사 VPN 접속 필수
- 신입사원: 입사 후 6개월간 재택 불가

[경비처리]
- 사용일로부터 14일 이내 영수증 제출
- 1회 5만원 초과 식대: 부서장 사전 승인 필요
- 출장 항공권: 이코노미 클래스 한정
- 개인 차량 업무 사용: km당 300원 유류비

[복리후생]
- 연 50만원 복지 포인트 (도서·교육·건강 항목만)
- 결혼 축하금 100만원 / 출산: 첫째 50만원, 둘째 100만원, 셋째 200만원
- 근속 3년마다 안식 휴가 1주일 + 여행 지원금 50만원

[보안]
- 사내 자료 외부 이메일·개인 클라우드 전송 금지
- 비밀번호: 90일마다 변경 / 영문+숫자+특수문자 10자 이상
- 회사 노트북 분실 시 24시간 이내 IT보안팀 신고

[평가/승진]
- 정기 평가: 연 2회 (상·하반기) / S·A·B·C·D 5단계
- 성과급: 평가 등급에 따라 기본급의 0~200%
- 승진 대상: 동일 직급 최소 3년 이상 근무
"""

print(f"✅ 내규 데이터 준비 완료 ({len(NOVACORP_RULES)}자)")

### 2-2. system 프롬프트 설계

EXAONE에게 **어떤 역할로 행동할지** 지시합니다.  
내규 데이터를 system 프롬프트에 포함시켜, 내규를 기반으로 답변하게 합니다.

In [ ]:
# system 프롬프트 — EXAONE의 역할과 행동 방식을 정의
SYSTEM_PROMPT = f"""당신은 노바코프(Novacorp)의 인사팀 AI 어시스턴트입니다.
직원들의 사내 내규 관련 질문에 정확하고 친절하게 답변하세요.

답변 규칙:
1. 아래 내규 내용만을 근거로 답변하세요
2. 내규에 없는 내용은 "해당 내용은 내규에 명시되어 있지 않습니다. 인사팀에 직접 문의하세요." 라고 안내하세요
3. 답변은 간결하게, 핵심만 전달하세요

=== 노바코프 사내 내규 ===
{NOVACORP_RULES}
"""

print(f"✅ system 프롬프트 준비 완료 ({len(SYSTEM_PROMPT)}자)")

### 2-3. 역할 테스트

system 프롬프트를 적용했을 때 실제로 내규 기반으로 답변하는지 확인합니다.

In [ ]:
# 내규 질문으로 테스트
role_test = generate_response(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": "연차는 몇 일이나 되나요?"},
    ],
    temperature=0.3,     # 정확성 우선 → 낮게
    max_new_tokens=300,
)

print("질문: 연차는 몇 일이나 되나요?")
print("응답:", role_test["content"])
print("\n✅ STAGE 2 완료: 노바코프 내규 기반 답변 확인!")
tracker.add(role_test)

---
# STAGE 3. 대화 기억 — 이전 대화를 기억하게 하기

지금은 매번 새로운 대화로 시작합니다.  
"아까 말한 내용 요약해줘" 하면 EXAONE이 모릅니다.

이전 대화를 **히스토리 리스트**에 쌓아서 EXAONE이 맥락을 알 수 있게 합니다.

### 3-1. 히스토리 리스트 초기화

대화 내용을 담을 **리스트**를 만듭니다.  
system 프롬프트로 시작합니다.

In [ ]:
# 대화 히스토리 — system 프롬프트로 초기화
history = [
    {"role": "system", "content": SYSTEM_PROMPT}   # 역할 지정
]

print(f"✅ 히스토리 초기화 완료")
print(f"   현재 항목 수: {len(history)}개 (system만 있음)")

### 3-2. chat() 함수 만들기

대화할 때마다 히스토리를 **쌓고 → 추론 호출 → 결과도 쌓는** 함수입니다.  
이 함수가 멀티턴 챗봇의 핵심입니다.

```
① user 메시지를 history에 추가
② history 전체를 모델에 전달  ← 이래야 맥락이 유지됩니다!
③ 응답을 history에 추가
④ 응답 반환
```

In [ ]:
# chat 함수 — 멀티턴 대화의 핵심
def chat(user_input):
    # ① user 메시지를 히스토리에 추가
    history.append({"role": "user", "content": user_input})

    # ② 히스토리 전체를 모델에 전달 (이전 대화가 모두 포함됨)
    result = generate_response(
        messages=history,    # 전체 대화 내용
        temperature=0.3,
        max_new_tokens=300,
    )

    # ③ 응답을 히스토리에 추가
    reply = result["content"]
    history.append({"role": "assistant", "content": reply})

    # ④ 토큰 추적 후 응답 반환
    tracker.add(result)
    return reply

print("✅ chat() 함수 준비 완료")

### 3-3. 맥락 유지 테스트

3턴 대화로 **이전 내용을 기억하는지** 확인합니다.  
3번째 질문에서 1·2번 내용이 담긴 요약이 나오면 성공입니다.

In [ ]:
# 3턴 대화 테스트
print("[1턴] 질문: 연차는 며칠이고, 어떻게 신청하나요?")
print("응답:", chat("연차는 며칠이고, 어떻게 신청하나요?"))

print("\n" + "─"*50)
print("[2턴] 질문: 재택근무는 주에 며칠까지 되나요?")
print("응답:", chat("재택근무는 주에 며칠까지 되나요?"))

print("\n" + "─"*50)
print("[3턴] 질문: 방금 알려준 내용을 한 줄씩 요약해줘")
print("응답:", chat("방금 알려준 내용을 한 줄씩 요약해줘"))   # ← 1·2턴 내용 참조 확인!

In [ ]:
# 히스토리가 올바르게 쌓였는지 확인
print(f"히스토리 항목 수: {len(history)}개")
print(f"예상: system(1) + 3턴 × 2 = 7개")
print()

# role 순서 확인
roles = [m["role"] for m in history]
print(f"role 순서: {roles}")
print("\n✅ STAGE 3 완료: 맥락 유지 확인!")

---
# STAGE 4. 안전 장치 — 실패해도 괜찮게 만들기

로컬 추론은 GPU 메모리 부족이나 토큰 초과 등으로 실패할 수 있습니다.  
실패했을 때 **히스토리가 망가지지 않도록** 안전 장치를 추가합니다.

### 4-1. 안전 추론 함수 만들기

실패하면 GPU 캐시를 정리하고 최대 2회 재시도합니다.

> 💡 **API vs 로컬 재시도 전략의 차이**  
> - API: 네트워크·Rate Limit 오류 → 지수 백오프 (1→2→4초 대기)  
> - 로컬: GPU 메모리 오류가 주원인 → **GPU 캐시 정리 후 즉시 재시도**

In [ ]:
# 안전한 추론 함수 — GPU 오류 시 캐시 정리 후 재시도
def safe_call(messages, temperature=0.3, max_new_tokens=300):
    """실패 시 GPU 캐시 정리 후 최대 2회 재시도. 모두 실패하면 None 반환."""

    for attempt in range(2):   # 최대 2번 시도
        try:
            result = generate_response(
                messages=messages,
                temperature=temperature,
                max_new_tokens=max_new_tokens,
            )
            tracker.add(result)          # 성공 → 토큰 기록
            return result["content"]     # 성공 → 응답 반환

        except Exception as e:
            print(f"  ⚠️ [{attempt+1}/2] 오류 발생 → GPU 메모리 정리 후 재시도")
            torch.cuda.empty_cache()

    return None   # 2번 모두 실패 → None 반환

print("✅ safe_call() 준비 완료")

### 4-2. 안전 버전 chat 함수로 업그레이드

기존 `chat()` 을 `chat_safe()` 로 교체합니다.  
실패하면 **히스토리를 되돌려서** 대화가 꼬이지 않게 합니다.

In [ ]:
# 안전한 버전의 chat 함수
def chat_safe(user_input):
    # ① user 메시지를 히스토리에 추가
    history.append({"role": "user", "content": user_input})

    # ② 안전하게 추론 호출
    reply = safe_call(history)

    if reply is None:
        # 실패 시: 방금 추가한 user 메시지를 제거 (히스토리 보호!)
        history.pop()
        return "⚠️ 응답을 받지 못했습니다. 잠시 후 다시 시도해 주세요."

    # ③ 응답을 히스토리에 추가
    history.append({"role": "assistant", "content": reply})
    return reply

print("✅ chat_safe() 준비 완료")

In [ ]:
# 히스토리 초기화 함수 — 새 대화를 시작할 때 사용
def reset():
    global history
    history = [{"role": "system", "content": SYSTEM_PROMPT}]
    print("🔄 대화가 초기화되었습니다. (새 대화 시작)")

# 안전 버전 테스트
reset()   # 히스토리 초기화 후 테스트
answer = chat_safe("결혼하면 어떤 혜택을 받을 수 있나요?")
print("질문: 결혼하면 어떤 혜택을 받을 수 있나요?")
print("응답:", answer)
print("\n✅ STAGE 4 완료: 안전 장치 확인!")

---
# STAGE 5. 챗봇 완성 — 실제로 대화해보기

STAGE 1~4에서 만든 모든 것을 합쳐서 **노바코프 내규 챗봇 v0.1** 을 완성합니다!

### 5-1. 챗봇 최종 점검

완성 전에 모든 부품이 준비됐는지 확인합니다.

In [ ]:
# 챗봇 완성 전 최종 점검
checks = {
    "모델 로드":        model is not None,
    "토크나이저 로드":   tokenizer is not None,
    "내규 데이터":       len(NOVACORP_RULES) > 0,
    "system 프롬프트":  len(SYSTEM_PROMPT) > 0,
    "chat_safe 함수":  callable(chat_safe),
    "reset 함수":      callable(reset),
    "토큰 추적기":       tracker is not None,
}

print("🔍 챗봇 완성 전 점검")
all_ok = True
for item, status in checks.items():
    icon = "✅" if status else "❌"
    print(f"  {icon} {item}")
    if not status:
        all_ok = False

print()
if all_ok:
    print("🎉 모든 점검 완료! 챗봇을 시작합니다.")
else:
    print("⚠️ 위의 ❌ 항목을 먼저 해결하세요.")

### 5-2. 노바코프 내규 챗봇 v0.1 실행!

실제 직원이 물어볼 법한 질문 5가지로 챗봇을 테스트합니다.  
마지막 질문은 **내규에 없는 내용**입니다 — 어떻게 답하는지 확인해보세요.

In [ ]:
# 새 대화로 시작
reset()

# 실제 직원 질문 시나리오
test_questions = [
    "안녕하세요! 저 이번에 입사한 신입사원인데요, 연차는 언제부터 쓸 수 있나요?",
    "그럼 재택근무는 바로 할 수 있나요?",          # 신입사원 + 재택 → 맥락 연결
    "출장 갈 때 비행기 비즈니스 타도 되나요?",
    "비밀번호는 어떻게 설정해야 하나요?",
    "야간 수당은 얼마나 받을 수 있나요?",           # 내규에 없는 내용 → 어떻게 답하나?
]

print("=" * 60)
print("🤖 노바코프 내규 챗봇 v0.1 (EXAONE 3.5)")
print("=" * 60)

for i, q in enumerate(test_questions, 1):
    print(f"\n[{i}번 질문] {q}")
    print("─" * 40)
    answer = chat_safe(q)
    print(f"챗봇: {answer}")

### 5-3. 오늘 사용한 토큰 최종 확인

In [ ]:
# 오늘 전체 실습 토큰 사용량 최종 확인
print("💰 오늘 실습 전체 토큰 사용량")
tracker.report()

print()
print("=" * 60)
print("🎉 노바코프 내규 챗봇 v0.1 완성!")
print("=" * 60)
print("""
오늘 만든 것:
  ✅ STAGE 1. EXAONE 모델 로드 + 추론 함수 준비
  ✅ STAGE 2. 노바코프 전문가로 역할 부여
  ✅ STAGE 3. 이전 대화 기억하는 멀티턴 구조
  ✅ STAGE 4. 실패해도 안전한 GPU 재시도 장치
  ✅ STAGE 5. 챗봇 완성 + 실전 테스트
""")

---
## 🔜 다음 시간 예고

> **오늘 발견한 문제:**  
> 같은 질문을 여러 번 하면 답변 품질이 조금씩 다릅니다.  
> 어떤 때는 내규를 구조적으로 설명하고, 어떤 때는 단순하게 답합니다.
>
> **Day 2에서 해결할 것:**  
> Zero-shot / Few-shot / CoT 프롬프트 기법으로  
> 이 품질 불일치를 줄일 수 있는지 실험합니다.
>
> **아직 해결 안 된다면?**  
> → Day 4 파인튜닝이 그 답입니다.

> 💡 EXAONE은 로컬 추론이므로 API 비용 없이 실험할 수 있습니다.  
> Day 4에서는 직접 **파인튜닝**으로 품질을 고정시킵니다!